# Bài tập 3 — Chọn ngưỡng theo yêu cầu recall

Tìm ngưỡng lớn nhất có recall cross-validation trên tập học ít nhất 0.80; sau đó đánh giá trên tập kiểm tra một lần.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(Path("../data/sinh_vien_bo_hoc.csv"))
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=17, stratify=y)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

lr = make_pipeline(StandardScaler(), LogisticRegression())
import numpy as np
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import cross_val_predict
xs_hoc=cross_val_predict(lr,X_tr,y_tr,cv=5,method="predict_proba")[:,1]
nguong=np.round(np.arange(0.01,1.00,0.01),2)
ketqua=[recall_score(y_tr,(xs_hoc>=t).astype(int)) for t in nguong]
dat=[(t,r) for t,r in zip(nguong,ketqua) if r>=0.80]
if not dat: raise ValueError("Khong tim thay nguong dat recall yeu cau")
t_tot=max(t for t,_ in dat)
lr.fit(X_tr,y_tr); du=(lr.predict_proba(X_te)[:,1]>=t_tot).astype(int)
print(f"Nguong lon nhat tren tap hoc: {t_tot:.2f}")
print(f"Test recall={recall_score(y_te,du):.4f}, precision={precision_score(y_te,du,zero_division=0):.4f}, so bi bao={du.sum()}")

**Kết luận:** recall trên test có thể không đạt 0.80 dù ngưỡng được chọn thỏa điều kiện trên dự đoán cross-validation của tập học. Đây là ước lượng thực tế về khả năng tổng quát hóa, không nên tiếp tục chỉnh ngưỡng theo test.